# Tokke–Vinje — caricamento e costruzione della tabella *long*

**Obiettivo di questo notebook:** leggere i file SINTEF, associare ogni generatore ai suoi reservoir e ottenere **una riga per (Run No, generatore, ora)**, con **un solo target binario** (`target`, ON/OFF).

Non alleniamo modelli, non facciamo tuning e non scegliamo feature in questa fase.

### Allineamento temporale (importante)
- Prezzo e inflow: all'ora `starttime + hour` (ore 0…167).
- Volume: **inizio della finestra**, cioè `starttime` per tutte le 168 ore del caso. Non usiamo volumi osservati durante i giorni futuri. Le nuove feature di topologia estesa utilizzano questi volumi iniziali e le capacità massime dello YAML.
- Water value: **fine della finestra**, cioè `starttime + 168 ore`, come richiesto dal documento SINTEF per un problema settimanale.
- Vincoli minimi: valore orario della riga; manteniamo tutti i 3 vincoli di volume e i 6 di portata, per decidere nell'EDA come usarli.
- `total_calculation_time`: **mai fra le feature**. Compare esclusivamente nella prova di ricostruzione del CSV originale.

**Attenzione:** il dataset contiene finestre settimanali che iniziano ogni giorno, quindi sono fortemente sovrapposte. Lo split train/validation/test andrà fatto **per tempo e per caso**, non casualmente sulle righe long. Prima dell'allenamento andrà verificato anche che gli inflow orari futuri siano effettivamente disponibili nello scenario di predizione.

Appunti di Ghigu: praticamente stiamo creando la nuova tabella, dove in ogni riga abbiamo un solo generatore. informazione come Run no ( identificativo= , startime, timestamp e cose del genere NON le passeremo come feature al modello, perchè sono inutili a scopo predittivo. Per ora però le teniamo in tabella perchè sono informazioni che ci serviranno poi per ricostruire il formato di dati originali da "consegnare"/"comparare" con il risultati del prof.

## 1 — Importazioni e funzioni di lettura

I CSV `Historical_inflow_1958_2025.csv` contengono moltissimi anni: ne carichiamo solo l'intervallo necessario. 

In [1]:
from __future__ import annotations

import argparse
from collections import defaultdict
from pathlib import Path
import re

import numpy as np
import pandas as pd
import yaml

# Numero di ore per ogni caso (7 giorni)
HOURS = 168

# Cartella principale della repository: funziona avviando Jupyter dalla root o da Ghigu/
WORKING_DIR = Path.cwd().resolve()
if (WORKING_DIR / "sintef_project" / "data" / "kernel").is_dir():
    PROJECT_DIR = WORKING_DIR
elif (WORKING_DIR.parent / "sintef_project" / "data" / "kernel").is_dir():
    PROJECT_DIR = WORKING_DIR.parent
else:
    raise FileNotFoundError(
        "Repository non trovata: avvia Jupyter da ml-project-ntnu/ o ml-project-ntnu/Ghigu/"
    )

# Directory contenenti i dati SINTEF
KERNEL_DIR = PROJECT_DIR / "sintef_project" / "data" / "kernel"
EXTENDED_DIR = PROJECT_DIR / "sintef_project" / "data" / "extended"

# Percorsi dei file originali
DATA_PATHS = {
    "decisions": KERNEL_DIR / "Unit_commitment_decisions.csv",
    "price": KERNEL_DIR / "Historical_day_ahead_price_2015_2025.csv",
    "inflow": KERNEL_DIR / "Historical_inflow_1958_2025.csv",
    "volume": KERNEL_DIR / "Historical_volume_2015_2024.csv",
    "water_value": KERNEL_DIR / "Synthetic_water_value_2015_2024.csv",
    "min_flow": EXTENDED_DIR / "Constraint_min_flow.csv",
    "min_volume": EXTENDED_DIR / "Constraint_min_volume.csv",
}

TOPOLOGY_PATH = EXTENDED_DIR / "Tokke_Vinje_topology.yaml"

'''
Esattamente! La funzione prende un file CSV ed estrae soltanto le righe comprese tra 
`min_time` e `max_time`, estremi inclusi.
In più, svolge tre piccole operazioni:
    - Converte le date in un formato temporale standard (UTC).
    - Controlla che non ci siano timestamp duplicati.
    - Converte i valori numerici in `float32` per risparmiare memoria.
Opzionalmente può anche selezionare soltanto alcune colonne.
Quindi sì: il suo scopo principale è filtrare le righe di un dataset in base a un intervallo 
temporale. Non modifica i valori originali, salvo la conversione del tipo numerico.

'''
def _read_time_table(
    path: Path,
    min_time: pd.Timestamp,
    max_time: pd.Timestamp,
    columns: list[str] | None = None
) -> pd.DataFrame:

    # Lettura del CSV
    df = pd.read_csv(path, usecols=columns)

    # Conversione delle date
    df["date"] = pd.to_datetime(df["date"], utc=True)

    # Selezione dell'intervallo temporale richiesto
    df = df.loc[df["date"].between(min_time, max_time)].set_index("date")

    # Controllo di eventuali timestamp duplicati
    if not df.index.is_unique:
        raise ValueError(f"Duplicate timestamps in {path}")

    return df.astype("float32")


'''
La funzione _read_inflow() serve a caricare i dati degli afflussi d'acqua (inflow) dei reservoir,
mantenendo soltanto le righe comprese nell'intervallo temporale che ci interessa.
Quindi fa sostanzialmente la stessa cosa della funzione _read_time_table() che abbiamo visto prima.

Perché abbiamo creato due funzioni diverse?
Perché il file Historical_inflow_1958_2025.csv è particolarmente grande: contiene dati orari dal 1958 al 2025.
Anziché caricare immediatamente tutto il file in memoria, _read_inflow() lo legge progressivamente, 
in blocchi da 100.000 righe.

'''
def _read_inflow(
    path: Path,
    min_time: pd.Timestamp,
    max_time: pd.Timestamp
) -> pd.DataFrame:

    # Lettura delle intestazioni
    columns = pd.read_csv(path, nrows=0).columns.tolist()

    # Utilizziamo float32 per ridurre il consumo di RAM
    dtypes = {c: "float32" for c in columns if c != "date"}

    parts = []

    # Lettura del CSV in blocchi da 100.000 righe
    for chunk in pd.read_csv(path, chunksize=100_000, dtype=dtypes):

        # Selezione delle righe nell'intervallo richiesto
        keep = chunk["date"].str[:10].between(
            min_time.strftime("%Y-%m-%d"),
            max_time.strftime("%Y-%m-%d")
        )

        subset = chunk.loc[keep].copy()

        if not subset.empty:
            subset["date"] = pd.to_datetime(subset["date"], utc=True)

            subset = subset.loc[
                subset["date"].between(min_time, max_time)
            ]

            parts.append(subset)

    if not parts:
        raise ValueError("No inflow data found for selected cases")

    # Unione dei blocchi selezionati
    inflow = pd.concat(parts, ignore_index=True).set_index("date")

    if not inflow.index.is_unique:
        raise ValueError("Duplicate timestamps in inflow")

    return inflow

## 2 — Lettura di tutti i dataset e controllo dei target

la prossima funzione praticamente convalida i dati e li carica in memoria, tutto sotto al Giga quindi chill 

In [2]:
def load_data(data_dir: Path, limit_cases: int | None = None) -> dict:
    """Load the original input tables and restrict time series to the actual cases."""

    paths = DATA_PATHS
    topo_path = TOPOLOGY_PATH
    
    all_columns = pd.read_csv(paths["decisions"], nrows=0).columns.tolist()
    target_cols = [c for c in all_columns if c.startswith("result_committed_")]
    decision_types = {c: "uint8" for c in target_cols}
    decision_types.update({"Run No": "int32", "total_calculation_time": "float32"})
    decisions = pd.read_csv(paths["decisions"], nrows=limit_cases, dtype=decision_types)
    decisions["starttime"] = pd.to_datetime(decisions["starttime"], utc=True)
    if not decisions["Run No"].is_unique:
        raise ValueError("Run No is not unique")

    with topo_path.open(encoding="utf-8") as file:
        topology = yaml.safe_load(file)

    generators = list(topology["model"]["generator"].keys())
    target_lookup = {
        gen: [f"result_committed_{gen}_t{h}" for h in range(HOURS)]
        for gen in generators
    }
    target_expected = {name for cols in target_lookup.values() for name in cols}
    if set(target_cols) != target_expected:
        missing = target_expected - set(target_cols)
        extra = set(target_cols) - target_expected
        raise ValueError(f"Target columns do not match YAML generators: missing={missing}, extra={extra}")
    if decisions[target_cols].isna().any().any():
        raise ValueError("Missing target values")
    if not decisions[target_cols].isin([0, 1]).all().all():
        raise ValueError("Targets must be 0 or 1")

    min_start = decisions["starttime"].min()
    max_hour = decisions["starttime"].max() + pd.Timedelta(hours=HOURS-1)
    max_end = decisions["starttime"].max() + pd.Timedelta(hours=HOURS)

    price = _read_time_table(paths["price"], min_start, max_hour)
    price = price.rename(columns={"Day-ahead price (EUR/MWh)": "price_current"})
    inflow = _read_inflow(paths["inflow"], min_start, max_hour)
    min_flow = _read_time_table(paths["min_flow"], min_start, max_hour)
    min_volume = _read_time_table(paths["min_volume"], min_start, max_hour)
    volume = _read_time_table(paths["volume"], min_start, decisions["starttime"].max())
    water_value = _read_time_table(paths["water_value"], min_start, max_end)

    if "price_current" not in price:
        raise ValueError("Price column missing")
    res_names = set(topology["model"]["reservoir"])
    for key, df in [("inflow", inflow), ("volume", volume), ("water_value", water_value)]:
        if not res_names.issubset(df.columns):
            raise ValueError(f"Missing reservoir columns in {key}: {sorted(res_names-set(df.columns))}")

    return dict(decisions=decisions, target_cols=target_cols, target_lookup=target_lookup,
                generators=generators, topology=topology, price=price, inflow=inflow,
                min_flow=min_flow, min_volume=min_volume, volume=volume, water_value=water_value)

## 3 — Mappatura generatori e reservoir (YAML)

`topology_metadata()` legge la topologia e identifica, per ogni generatore, i reservoir **direct** e **upstream** (questi ultimi **includono** i direct). Manteniamo invariata la logica precedente di riconoscimento dei collegamenti e le caratteristiche tecniche del generatore. La mappatura non costruisce ancora le colonne numeriche delle feature.

Inoltre ricaviamo i reservoir **upstream-only** (upstream senza i direct) e i **primi reservoir a valle** seguendo il verso delle connessioni. Il percorso downstream può attraversare nodi `river` e `tunnel`, ma si ferma al primo reservoir; per esempio `Lio → r_Vest_Vassdraget → Bandak`. `Hogga` non ha un reservoir ricevente nel grafo.


In [3]:
def topology_metadata(topology: dict) -> pd.DataFrame:
    """Direct reservoirs and ALL upstream reservoirs (direct included).

    Follows directed YAML edges, including tunnels and rivers. Reservoirs reached via
    different paths are counted only once. In particular Songa has two direct reservoirs.
    """
    predecessors = defaultdict(list)
    successors = defaultdict(list)
    for edge in topology["connections"]:
        a = (edge["from_type"], edge["from"])
        b = (edge["to_type"], edge["to"])
        predecessors[b].append(a)
        successors[a].append(b)

    def first_downstream_reservoirs(plant: str) -> list[str]:
        """Reservoir raggiunti dall'uscita della centrale, anche tramite fiumi/tunnel.

        Il percorso si ferma al PRIMO reservoir: non attraversa reservoir
        successivi né altre centrali (che rappresenterebbero stati più a valle).
        """
        receivers = set()
        visited = set()
        stack = [("plant", plant)]
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            if node[0] == "reservoir":
                receivers.add(node[1])
                continue
            if node[0] not in ("plant", "river", "tunnel"):
                continue
            for next_node in successors[node]:
                if next_node[0] in ("reservoir", "river", "tunnel"):
                    stack.append(next_node)
        return sorted(receivers)

    plant_of_gen = {
        edge["from"]: edge["to"]
        for edge in topology["connections"]
        if edge["from_type"] == "generator" and edge["to_type"] == "plant"
    }
    capacities = {res: value["max_vol"] for res, value in topology["model"]["reservoir"].items()}
    records = []
    for gen, spec in topology["model"]["generator"].items():
        plant = plant_of_gen[gen]
        direct = set()
        for node_type, node_name in predecessors[("plant", plant)]:
            if node_type == "reservoir":
                direct.add(node_name)
            if node_type == "tunnel":
                direct.update(name for kind, name in predecessors[("tunnel", node_name)]
                              if kind == "reservoir")

        visited = set()
        stack = [("plant", plant)]
        upstream = set()
        while stack:
            node = stack.pop()
            if node in visited:
                continue
            visited.add(node)
            if node[0] == "reservoir":
                upstream.add(node[1])
            stack.extend(predecessors[node])

        if not direct or not direct.issubset(upstream):
            raise ValueError(f"Bad reservoir mapping for {gen}: {direct}, {upstream}")

        # Upstream comprende i direct; qui li separiamo per le nuove feature.
        upstream_only = upstream - direct
        downstream = first_downstream_reservoirs(plant)
        records.append({
            "generator": gen, "plant": plant,
            "direct_reservoirs": sorted(direct),
            "upstream_reservoirs": sorted(upstream),
            "upstream_only_reservoirs": sorted(upstream_only),
            "downstream_reservoirs": downstream,
            "direct_capacity": sum(capacities[r] for r in direct),
            "upstream_capacity": sum(capacities[r] for r in upstream),
            "downstream_capacity": sum(capacities[r] for r in downstream),
            "p_min": float(spec["p_min"]), "p_max": float(spec["p_max"]),
            "p_nom": float(spec["p_nom"]),
            "start_cost": float(spec["startcost_const"]),
            "stop_cost": float(spec["stopcost_const"]),
            "penstock": int(spec["penstock"]),
        })
    return pd.DataFrame(records).set_index("generator", drop=False)


## 4 — Feature engineering: topologia

Questa sezione **calcola le 7 feature topologiche** già presenti nella versione originale, senza modificarne le formule:

- **Direct:** `direct_total_volume`, `direct_mean_fill_ratio`, `direct_total_inflow`, `direct_mean_water_value`.
- **Upstream:** `upstream_total_volume`, `upstream_total_inflow`, `upstream_mean_water_value`.

I volumi e i water value sono riferiti rispettivamente all'inizio e alla fine del caso settimanale, come nella versione originale; gli inflow sono orari. I water value sono mediati con pesi pari ai volumi iniziali e `upstream` comprende i reservoir `direct`. La funzione restituisce un **dizionario nome-colonna → valori** che verrà aggiunto a `gen_cols` in `make_batch()`.


In [4]:
def _weighted(values: pd.DataFrame, volumes: pd.DataFrame, selected: list[str]) -> np.ndarray:
    """Marginal water value weighted by initial reservoir volumes (zero total -> NaN)."""
    v = volumes[selected].to_numpy(dtype="float64")
    x = values[selected].to_numpy(dtype="float64")
    num = (v * x).sum(axis=1)
    denom = v.sum(axis=1)
    return np.divide(num, denom, out=np.full(len(denom), np.nan), where=denom > 0).astype("float32")


def compute_topology_features(
    m: pd.Series,
    initial_volumes: pd.DataFrame,
    final_water_value: pd.DataFrame,
    hourly_inflow: pd.DataFrame,
) -> dict[str, np.ndarray]:
    """Calcola le sette feature topologiche per un generatore e tutti i casi del batch."""
    direct = m["direct_reservoirs"]
    upstream = m["upstream_reservoirs"]  # upstream INCLUDE direct

    # Calcoli relativi ai reservoir directly connected
    direct_volume = initial_volumes[direct].sum(axis=1).to_numpy(dtype="float32")
    direct_water = _weighted(final_water_value, initial_volumes, direct)
    direct_inflow = hourly_inflow[direct].sum(axis=1).to_numpy(dtype="float32")

    # Calcoli relativi ai reservoir upstream (inclusi i direct)
    upstream_volume = initial_volumes[upstream].sum(axis=1).to_numpy(dtype="float32")
    upstream_water = _weighted(final_water_value, initial_volumes, upstream)
    upstream_inflow = hourly_inflow[upstream].sum(axis=1).to_numpy(dtype="float32")

    return {
        # DIRECT topology features
        "direct_total_volume": np.repeat(direct_volume, HOURS),
        # Capacity-weighted mean fill ratio, not an unweighted mean of ratios.
        "direct_mean_fill_ratio": np.repeat(
            direct_volume / m["direct_capacity"], HOURS
        ).astype("float32"),
        "direct_total_inflow": direct_inflow,
        "direct_mean_water_value": np.repeat(direct_water, HOURS),

        # UPSTREAM topology features
        "upstream_total_volume": np.repeat(upstream_volume, HOURS),
        "upstream_total_inflow": upstream_inflow,
        "upstream_mean_water_value": np.repeat(upstream_water, HOURS),
    }


## 5 — Feature engineering: prezzo dell'elettricità

Da `price_current` (prezzo day-ahead originale) calcoliamo **9 nuove feature**, per ogni finestra di 168 ore del singolo caso:

| Feature | Significato |
|---|---|
| `price_mean_week` | Media dei 168 prezzi |
| `price_std_week` | Deviazione standard dei 168 prezzi (popolazione, `ddof=0`) |
| `price_min_week` | Minimo delle 168 ore |
| `price_max_week` | Massimo delle 168 ore |
| `price_diff_week_mean` | Prezzo corrente meno media settimanale |
| `price_rank_week` | Percentile del rango orario nella settimana, tra `1/168` e `1` (pareggi con rango medio) |
| `price_delta_prev_hour` | Prezzo attuale meno prezzo dell'ora precedente **nello stesso caso** |
| `price_mean_next_3h` | Media delle prossime fino a 3 ore **esclusa quella corrente** |
| `price_mean_next_12h` | Media delle prossime fino a 12 ore **esclusa quella corrente** |

**Gestione dei bordi:** per l'ora 0 non esiste un'ora precedente *nel caso* e `price_delta_prev_hour` vale `NaN`; per l'ora 167 non esistono ore successive *nel caso* e le due medie future valgono `NaN`. Negli ultimi passi consideriamo solo le ore future ancora disponibili nel caso, senza invadere quello successivo. XGBoost sa trattare i `NaN`; eventuali altri modelli dovranno gestirli esplicitamente.

**Disponibilità dei dati:** le statistiche sull'intera finestra e le medie future sono lecite soltanto se i **168 prezzi day-ahead del caso sono noti al momento della previsione**. Verificare questa condizione per i dati di test. Queste feature sono condivise tra generatori, quindi saranno calcolate **una sola volta per batch** e aggiunte al dizionario `common`.


In [5]:
def compute_price_features(hourly_price: pd.Series) -> dict[str, np.ndarray]:
    """Costruisce nove feature dei prezzi, per caso e ora, allineate al formato long.

    hourly_price contiene n_casi * HOURS valori ordinati per caso e ora.
    Nessuna finestra temporale attraversa il confine tra due casi.
    """
    prices = hourly_price.to_numpy(dtype="float32").reshape(-1, HOURS)
    n_cases = len(prices)

    # Statistiche della settimana: un valore per caso, ripetuto sulle 168 ore.
    weekly_mean = prices.mean(axis=1, dtype="float64").astype("float32")
    weekly_std = prices.std(axis=1, ddof=0, dtype="float64").astype("float32")
    weekly_min = prices.min(axis=1)
    weekly_max = prices.max(axis=1)

    # Le differenze e il rango variano da un'ora all'altra.
    price_diff = prices - weekly_mean[:, None]
    ranks = pd.DataFrame(prices).rank(axis=1, method="average", pct=True).to_numpy(dtype="float32")

    # t=0: ora precedente assente all'interno del caso.
    delta_prev = np.full((n_cases, HOURS), np.nan, dtype="float32")
    delta_prev[:, 1:] = prices[:, 1:] - prices[:, :-1]

    # Le finestre future escludono l'ora corrente e non oltrepassano l'ora 167.
    # t=167: non ci sono ore future -> NaN, anziché introdurre un valore arbitrario.
    sums = np.concatenate(
        [np.zeros((n_cases, 1), dtype="float64"), np.cumsum(prices, axis=1, dtype="float64")],
        axis=1,
    )

    def next_hours_mean(horizon: int) -> np.ndarray:
        first = np.arange(HOURS, dtype="int32") + 1
        last = np.minimum(first + horizon, HOURS)
        count = last - first
        window_sum = sums[:, last] - sums[:, first]
        result = np.full((n_cases, HOURS), np.nan, dtype="float64")
        np.divide(window_sum, count, out=result, where=count > 0)
        return result.astype("float32")

    return {
        # WEEKLY price statistics
        "price_mean_week": np.repeat(weekly_mean, HOURS),
        "price_std_week": np.repeat(weekly_std, HOURS),
        "price_min_week": np.repeat(weekly_min, HOURS),
        "price_max_week": np.repeat(weekly_max, HOURS),

        # HOURLY price features
        "price_diff_week_mean": price_diff.reshape(-1),
        "price_rank_week": ranks.reshape(-1),
        "price_delta_prev_hour": delta_prev.reshape(-1),
        "price_mean_next_3h": next_hours_mean(3).reshape(-1),
        "price_mean_next_12h": next_hours_mean(12).reshape(-1),
    }


## 6 — Feature engineering: relazione prezzo elettricità–water value

Questa cella calcola **3 feature aggiuntive**, specifiche di ogni generatore, usando `direct_mean_water_value` già calcolato nella sezione topologica e le colonne prezzo già preparate nella sezione elettricità. Non cambia il calcolo del water value: rimane la media pesata dei reservoir **direct**, riferita a `t_end` come nel notebook precedente.

- `price_minus_water_value`: prezzo orario meno water value diretto.
- `price_mean_week_minus_water_value`: prezzo medio delle 168 ore meno water value diretto.
- `price_max_week_minus_water_value`: massimo dei 168 prezzi meno water value diretto.

Queste colonne condividono il medesimo allineamento `(caso, ora)` delle altre feature. Non introduciamo nuove letture dei CSV né nuove assunzioni sulla disponibilità del water value (già verificata nel progetto). La sottrazione ha un'interpretazione economica diretta **solo se** prezzo e water value hanno unità compatibili; altrimenti rimane una feature numerica da validare empiricamente.



In [6]:
def compute_water_price_features(
    common: dict,
    topology_features: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    """Tre differenze tra prezzi elettrici e water value direct del generatore."""
    direct_water_value = topology_features["direct_mean_water_value"]

    return {
        "price_minus_water_value": (
            common["price_current"] - direct_water_value
        ).astype("float32"),
        "price_mean_week_minus_water_value": (
            common["price_mean_week"] - direct_water_value
        ).astype("float32"),
        "price_max_week_minus_water_value": (
            common["price_max_week"] - direct_water_value
        ).astype("float32"),
    }


## 7 — Feature engineering: topologia estesa (upstream-only / downstream)

Questa sezione aggiunge **4 feature** alle 7 feature topologiche storiche (che rimangono intatte):

- **`upstream_mean_fill_ratio`**: somma dei volumi iniziali di **tutti** i reservoir a monte, inclusi i direct, divisa per la somma delle loro capacità massime (`max_vol` dello YAML). È un rapporto pesato per capacità, non una media aritmetica dei fill ratio.
- **`upstream_only_mean_water_value`**: media dei water value a `starttime + 168 ore`, pesata per i volumi iniziali, dei soli reservoir **upstream ma non direct**.
- **`downstream_free_capacity_ratio`**: spazio iniziale disponibile nei **primi reservoir riceventi a valle**, diviso per la loro capacità massima aggregata, cioè `(capacità_max − volume_iniziale) / capacità_max`. Se il volume supera la capacità nominale la feature può essere negativa: non la limitiamo artificiosamente a zero.
- **`downstream_water_value`**: water value a fine finestra del/i primo/i reservoir riceventi a valle, pesato per i volumi iniziali se ve ne sono più di uno.

I collegamenti provengono **esclusivamente dalla topologia YAML**, senza inventare una mappatura centrale → reservoir; per esempio `Lio` raggiunge `Bandak` passando per `r_Vest_Vassdraget`. In questo YAML ogni centrale con uno sbocco in un reservoir ne ha **uno solo**. `Hogga` scarica fuori dal sistema rappresentato e non ha un reservoir downstream nello YAML: entrambe le sue nuove feature downstream diventano `NaN`. `Byrte` e `Songa` non hanno reservoir upstream-only: per esse `upstream_only_mean_water_value = NaN`. Anche una media pesata il cui peso totale è zero diventa `NaN`, non uno zero inventato.

Come per le feature esistenti, **volume e capacità** sono associati all'inizio delle 168 ore; **water value** usa il valore a fine finestra, secondo la convenzione già adottata. Questi valori sono ripetuti per le 168 ore del caso. I `NaN` delle combinazioni fisicamente non definite possono essere trattati direttamente da XGBoost, senza aggiungere colonne di indicatori mancanti.


In [7]:
def compute_extended_topology_features(
    m: pd.Series,
    initial_volumes: pd.DataFrame,
    final_water_value: pd.DataFrame,
) -> dict[str, np.ndarray]:
    """Quattro nuove feature di rete per generatore e per caso (168 ore)."""
    n_cases = len(initial_volumes)
    upstream = m["upstream_reservoirs"]  # INCLUDE i direct
    upstream_only = m["upstream_only_reservoirs"]
    downstream = m["downstream_reservoirs"]

    # ========================================================
    # UPSTREAM - riempimento di TUTTI i reservoir a monte
    # ========================================================
    upstream_volume = initial_volumes[upstream].sum(axis=1).to_numpy(dtype="float64")
    upstream_capacity = float(m["upstream_capacity"])
    if upstream_capacity <= 0:
        raise ValueError(f"Capacità upstream non valida per {m['generator']}")
    upstream_fill_ratio = (upstream_volume / upstream_capacity).astype("float32")

    # ========================================================
    # UPSTREAM-ONLY - water value SENZA reservoir direct
    # ========================================================
    if upstream_only:
        upstream_only_water = _weighted(final_water_value, initial_volumes, upstream_only)
    else:
        upstream_only_water = np.full(n_cases, np.nan, dtype="float32")

    # ========================================================
    # DOWNSTREAM - primo/i reservoir riceventi a valle
    # ========================================================
    if downstream:
        downstream_capacity = float(m["downstream_capacity"])
        if downstream_capacity <= 0:
            raise ValueError(f"Capacità downstream non valida per {m['generator']}")
        downstream_volume = initial_volumes[downstream].sum(axis=1).to_numpy(dtype="float64")
        free_capacity_ratio = ((downstream_capacity - downstream_volume) / downstream_capacity).astype("float32")
        downstream_water = _weighted(final_water_value, initial_volumes, downstream)
    else:
        # Nessun reservoir ricevente nello YAML (Hogga): informazione assente, non zero.
        free_capacity_ratio = np.full(n_cases, np.nan, dtype="float32")
        downstream_water = np.full(n_cases, np.nan, dtype="float32")

    return {
        "upstream_mean_fill_ratio": np.repeat(upstream_fill_ratio, HOURS),
        "upstream_only_mean_water_value": np.repeat(upstream_only_water, HOURS),
        "downstream_free_capacity_ratio": np.repeat(free_capacity_ratio, HOURS),
        "downstream_water_value": np.repeat(downstream_water, HOURS),
    }


## 8 — Costruzione del dataset long: `make_batch()`

`make_batch()` costruisce una riga per `(run_no, generator, hour)` elaborando un batch di casi alla volta:

1. **`common` — dati originali**: ID, prezzo orario e vincoli idrici.
2. **`common` — feature sui prezzi**: `compute_price_features()` (una sola volta per batch).
3. **`gen_cols` — dati originali del generatore**: potenze, costi, centrale e penstock.
4. **`gen_cols` — 7 feature topologiche esistenti**: `compute_topology_features()`.
5. **`gen_cols` — differenze prezzo–water value**: `compute_water_price_features()`.
6. **`gen_cols` — 4 feature di topologia estesa**: `compute_extended_topology_features()` (upstream fill, upstream-only water value, downstream spazio libero, downstream water value).
7. **`gen_cols` — target ON/OFF originale**.

Le vecchie feature del salto idraulico `reservoir_level` e `estimated_hydraulic_head` **non vengono più generate**. Le feature temporali e quelle relative all'inflow settimanale/storico erano già state rimosse. Manteniamo nel Parquet tutte le nove feature elettriche, anche quelle attualmente disabilitate nei modelli 03 e 04, per consentire esperimenti successivi.


In [8]:
def make_batch(data: dict, metadata: pd.DataFrame, cases: pd.DataFrame) -> pd.DataFrame:
    """Produce un batch long senza materializzare l'intero dataset in RAM."""
    n = len(cases)
    hours = np.tile(np.arange(HOURS, dtype="uint8"), n)
    starts = pd.DatetimeIndex(np.repeat(cases["starttime"].to_numpy(), HOURS))
    times = starts + pd.to_timedelta(hours.astype("int16"), unit="h")
    t0 = pd.DatetimeIndex(cases["starttime"])
    t_end = t0 + pd.Timedelta(hours=HOURS)

    # ============================================================
    # ORIGINAL INPUTS - Caricamento e allineamento temporale
    # ============================================================
    hourly_price = data["price"].reindex(times)["price_current"]
    hourly_inflow = data["inflow"].reindex(times)
    hourly_min_flow = data["min_flow"].reindex(times)
    hourly_min_volume = data["min_volume"].reindex(times)
    initial_volumes = data["volume"].reindex(t0)
    final_water_value = data["water_value"].reindex(t_end)
    sources = [hourly_price, hourly_inflow, hourly_min_flow,
               hourly_min_volume, initial_volumes, final_water_value]
    if any(source.isna().any().any() for source in sources):
        raise ValueError("Time alignment produced missing values; inspect input dates before proceeding")

    # ============================================================
    # COMMON - Original features (shared across generators)
    # ============================================================
    common = {
        "run_no": np.repeat(cases["Run No"].to_numpy(dtype="int32"), HOURS),
        "starttime": starts,
        "hour": hours,
        "timestamp": times,
        "price_current": hourly_price.to_numpy(dtype="float32"),
    }
    for column in hourly_min_volume.columns:
        common[f"min_volume_{column}"] = hourly_min_volume[column].to_numpy(dtype="float32")
    for column in hourly_min_flow.columns:
        common[f"min_flow_{column}"] = hourly_min_flow[column].to_numpy(dtype="float32")
    # Only river with exogenous inflow: keep as a separate global input.
    common["inflow_r_Vest_Vassdraget"] = hourly_inflow["r_Vest_Vassdraget"].to_numpy(dtype="float32")

    # ============================================================
    # FEATURE ENGINEERING - ELECTRICITY PRICE (shared)
    # ============================================================
    common.update(compute_price_features(hourly_price))

    frames = []
    for gen in data["generators"]:
        m = metadata.loc[gen]

        # ========================================================
        # GEN_COLS - Original generator characteristics
        # ========================================================
        gen_cols = {
            "generator": gen,
            "plant": m["plant"],
            "p_min": np.float32(m["p_min"]),
            "p_max": np.float32(m["p_max"]),
            "p_nom": np.float32(m["p_nom"]),
            "start_cost": np.float32(m["start_cost"]),
            "stop_cost": np.float32(m["stop_cost"]),
            "penstock": np.uint8(m["penstock"]),
        }

        # ========================================================
        # FEATURE ENGINEERING - TOPOLOGY (generator-specific)
        # ========================================================
        topology_features = compute_topology_features(
            m, initial_volumes, final_water_value, hourly_inflow
        )
        gen_cols.update(topology_features)

        # ========================================================
        # FEATURE ENGINEERING - ELECTRICITY PRICE / WATER VALUE
        # ========================================================
        gen_cols.update(compute_water_price_features(common, topology_features))

        # ========================================================
        # FEATURE ENGINEERING - EXTENDED TOPOLOGY (upstream-only / downstream)
        # ========================================================
        gen_cols.update(compute_extended_topology_features(
            m, initial_volumes, final_water_value
        ))

        # ========================================================
        # TARGET - Original binary ON/OFF commitment
        # ========================================================
        gen_cols["target"] = cases[data["target_lookup"][gen]].to_numpy(dtype="uint8").reshape(-1)

        frames.append(pd.DataFrame({**common, **gen_cols}))

    batch = pd.concat(frames, ignore_index=True)
    # Ordering isn't necessary for ML, but makes the file easy to inspect and audit.
    gen_order = pd.Categorical(batch["generator"], categories=data["generators"], ordered=True)
    batch = batch.assign(_gen_order=gen_order).sort_values(
        ["run_no", "_gen_order", "hour"], kind="stable"
    ).drop(columns="_gen_order").reset_index(drop=True)
    batch["generator"] = pd.Categorical(batch["generator"], categories=data["generators"])
    batch["plant"] = batch["plant"].astype("category")
    if batch.duplicated(["run_no", "generator", "hour"]).any():
        raise ValueError("Long keys are not unique")
    if len(batch) != n * len(data["generators"]) * HOURS:
        raise ValueError("Unexpected number of rows")
    return batch

## 9 — Ricostruzione inversa long → wide

Questa funzione è già pronta per la fase finale: potrà ricevere `value_column="prediction"` invece di `target`. Mantiene l'**ordine delle 2.352 colonne originali**. Il tempo di calcolo della soluzione è disponibile solo nel dataset di train, se passato separatamente come metadata.

difatto quesra funzione rimane anche se aggiungeremo nuove feature, percè al di la del fatto che le feature le diamo solo al modello, questa funzione usa solo 4 colonne 

run_no, generator, hour, target

che andranno a ricostruire il modello come vuole il prof 

Per esempio, partiamo da:
run_no	    generator	hour	price	fill_ratio	target
1	        Tokke_G1	0	    5	    0.72	    1
1	        Tokke_G1	1	    42	    0.72	    0
1	        okke_G2     0	    35	    0.65	    1

La parte 5 ricostruisce:

Run No	    result_committed_Tokke_G1_t0	result_committed_Tokke_G1_t1	result_committed_Tokke_G2_t0 ---
1	        1	                            0	                               1                         ---


In [9]:
def to_wide(long_table: pd.DataFrame, target_columns: list[str],
            value_column: str = "target", metadata: pd.DataFrame | None = None) -> pd.DataFrame:
    """Invert long predictions back into original 2352-column format (no training here).

    For a submission use value_column='prediction', after attaching predictions.
    total_calculation_time is never a feature; supply metadata only when comparing
    back to the training CSV. It will be NaN for new/test cases without this metadata.
    """
    key = ["run_no", "starttime", "generator", "hour"]
    small = long_table[key + [value_column]].copy()
    if small.duplicated(key).any():
        raise ValueError("Duplicated prediction keys")
    small["target_column"] = (
        "result_committed_" + small["generator"].astype(str)
        + "_t" + small["hour"].astype(str)
    )
    wide = small.pivot(index=["run_no", "starttime"],
                       columns="target_column", values=value_column)
    if not set(target_columns).issubset(wide.columns):
        raise ValueError("Missing predictions for at least one generator-hour column")
    wide = wide.reindex(columns=target_columns).reset_index()
    wide = wide.rename(columns={"run_no": "Run No", "starttime": "starttime"})
    if metadata is not None:
        calculation = metadata[["Run No", "total_calculation_time"]].copy()
        wide = wide.merge(calculation, on="Run No", validate="one_to_one")
    else:
        wide["total_calculation_time"] = np.nan
    return wide[["Run No", "starttime", "total_calculation_time"] + target_columns]

## 10 — Scrittura su disco

Questa sezione usa `make_batch()` per costruire progressivamente il dataset long da circa 6,9 milioni di righe e salvarlo su disco, senza mantenere tutti i casi simultaneamente in RAM.

Richiede `pyarrow` per Parquet: `pip install pandas numpy pyyaml pyarrow`.

In [10]:
def build_dataset(data_dir: Path, output_file: Path, *, limit_cases: int | None = None,
                  batch_cases: int = 32, output_format: str = "parquet") -> dict:
    """Write the data incrementally. Parquet recommended; CSV fallback for environments without pyarrow."""
    data = load_data(data_dir, limit_cases=limit_cases)
    mapping = topology_metadata(data["topology"])
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    if batch_cases < 1:
        raise ValueError("batch_cases must be positive")

    writer = None
    if output_format == "parquet":
        try:
            import pyarrow as pa
            import pyarrow.parquet as pq
        except ImportError as exc:
            raise ImportError("Parquet requires 'pip install pyarrow'. Alternatively use --format csv") from exc
    elif output_format != "csv":
        raise ValueError("output_format must be 'parquet' or 'csv'")

    rows = 0
    sample = None
    try:
        for start in range(0, len(data["decisions"]), batch_cases):
            cases = data["decisions"].iloc[start:start + batch_cases]
            chunk = make_batch(data, mapping, cases)
            if sample is None:
                sample = chunk.head(5)
                # Exact round trip check on one batch, including original column order.
                restored = to_wide(chunk, data["target_cols"], metadata=cases)
                expected = cases[["Run No", "starttime", "total_calculation_time"] + data["target_cols"]]
                pd.testing.assert_frame_equal(restored.reset_index(drop=True),
                                              expected.reset_index(drop=True), check_dtype=False,
                                              check_names=False)
            if output_format == "parquet":
                tbl = pa.Table.from_pandas(chunk, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(output_file, tbl.schema, compression="zstd")
                writer.write_table(tbl)
            else:
                chunk.to_csv(output_file, mode="w" if start == 0 else "a",
                             header=(start == 0), index=False)
            rows += len(chunk)
    finally:
        if writer is not None:
            writer.close()

    return {"rows": rows, "cases": len(data["decisions"]),
            "generators": len(data["generators"]), "features": sample.columns.tolist(),
            "sample": sample, "generator_mapping": mapping,
            "output": output_file}

## 11 — Generazione e salvataggio del dataset long

Questa sezione **esegue l'intera pipeline** definita nelle parti precedenti: carica i dataset originali, ricostruisce le associazioni tra generatori e reservoir, calcola le feature e genera la tabella in formato **long**.

La tabella viene costruita **a blocchi di 32 casi** (`BATCH_CASES = 32`), evitando di mantenere contemporaneamente tutte le righe in memoria.

### Configurazione

- `RUN_FULL_BUILD = True`: avvia la generazione e il salvataggio del dataset. lo metti false se stai testando codice e non vuoi processare 7 milioni di righe ( letteralmente lol)
- `LIMIT_CASES = None`: elabora tutti i casi disponibili. Imposta `2` per effettuare una prova sui primi due casi.
- `BATCH_CASES = 32`: numero di casi elaborati per ogni blocco.
- `OUTPUT_FORMAT = "parquet"`: formato del file generato. È possibile utilizzare anche `"csv"`.

### Risultato

Il dataset completo conterrà **6.872.544 righe**, una per ogni combinazione di caso, generatore e ora, con **un unico target binario ON/OFF** e le feature costruite nelle fasi precedenti.

Il file verrà salvato in:

`Ghigu/data/processed/feature_table_long.parquet`

Una volta generato, potrà essere caricato nei notebook successivi per eseguire EDA, feature engineering e training, senza ripetere ogni volta la costruzione della tabella.

**Consiglio:** eseguire inizialmente una prova con `LIMIT_CASES = 2`, verificare il risultato e successivamente impostare `None` per generare il dataset completo.

In [11]:
# 11 — Generazione del dataset long completo

# Cartella nella quale salvare il dataset elaborato
OUTPUT_DIR = PROJECT_DIR / "Ghigu" / "data" / "processed"

# Creiamo la cartella se non esiste
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configurazione
RUN_FULL_BUILD = True
LIMIT_CASES = None
BATCH_CASES = 32
OUTPUT_FORMAT = "parquet"

if RUN_FULL_BUILD:

    suffix = ".parquet" if OUTPUT_FORMAT == "parquet" else ".csv"

    output_file = OUTPUT_DIR / ("feature_table_long" + suffix)

    result = build_dataset(
        PROJECT_DIR,
        output_file,
        limit_cases=LIMIT_CASES,
        batch_cases=BATCH_CASES,
        output_format=OUTPUT_FORMAT
    )

    print(
        f"Creato {result['output']}: {result['rows']:,} righe, "
        f"{len(result['features'])} colonne"
    )

else:
    print("Generazione del dataset disabilitata.")

Creato /home/ghigu/Scrivania/ml-project-ntnu/Ghigu/data/processed/feature_table_long.parquet: 6,872,544 righe, 47 colonne


Parquet (.parquet) è semplicemente un formato per salvare tabelle di dati, proprio come CSV (.csv).
La differenza è che Parquet è progettato per gestire grandi quantità di dati in modo più efficiente.

## Schema del dataset long generato

Il Parquet contiene **47 colonne**: 31 colonne della struttura originale, 9 feature sui prezzi, 3 interazioni prezzo–water value e **4 nuove feature topologiche**. Le **7 feature topologiche preesistenti** sono comprese nelle 31 colonne originali. Rimangono invariati identificativi e target.

- **Identificativi:** `run_no`, `starttime`, `hour`, `timestamp`, `generator`, `plant`.
- **Prezzo originale:** `price_current`.
- **Feature elettriche:** `price_mean_week`, `price_std_week`, `price_min_week`, `price_max_week`, `price_diff_week_mean`, `price_rank_week`, `price_delta_prev_hour`, `price_mean_next_3h`, `price_mean_next_12h`.
- **Caratteristiche generatore:** `p_min`, `p_max`, `p_nom`, `start_cost`, `stop_cost`, `penstock`.
- **Interazioni prezzo–water value:** `price_minus_water_value`, `price_mean_week_minus_water_value`, `price_max_week_minus_water_value`.
- **Feature topologiche originarie:** `direct_total_volume`, `direct_mean_fill_ratio`, `direct_total_inflow`, `direct_mean_water_value`, `upstream_total_volume`, `upstream_total_inflow`, `upstream_mean_water_value`.
- **4 feature di topologia estesa:** `upstream_mean_fill_ratio`, `upstream_only_mean_water_value`, `downstream_free_capacity_ratio`, `downstream_water_value`.
- **Vincoli e inflow esterno:** le colonne `min_volume_*`, `min_flow_*`, `inflow_r_Vest_Vassdraget` già esistenti.
- **Target:** `target` (0=OFF, 1=ON).

Le feature idrauliche (`reservoir_level`, `estimated_hydraulic_head`), temporali e quelle di inflow settimanale/storico non selezionate sono **assenti**. Le quattro nuove feature possono contenere `NaN` dove non esistono reservoir compatibili nella rete (es. Hogga, Byrte, Songa): non si tratta di dati mancanti nei CSV.

La funzione `to_wide()` resta invariata. Il dataset viene scritto in `Ghigu/data/processed/feature_table_long.parquet`; prima di usare le nuove colonne nei notebook dei modelli, rigenerare il Parquet con questa versione del notebook.
